# Tutorial 5: MLPs and Features

**Prerequisites:** T01 (Transformers), T02 (Hooks), T03 (Residual Stream)

**What you'll learn:**
- How MLP layers store and retrieve knowledge
- What individual MLP neurons compute
- The concept of "features" and why neurons are often polysemantic
- How to trace a neuron's effect on the model's output

---

## MLPs: The Knowledge Banks

While attention heads **move information between positions**, MLP layers **transform information at each position independently**. Each MLP layer is a simple two-layer network:

```
MLP(x) = activation_fn(x @ W_in + b_in) @ W_out + b_out
```

- `W_in` expands from `d_model` to `d_mlp` (typically 4× larger)
- The activation function (GELU, ReLU, etc.) creates a sparse, nonlinear representation
- `W_out` projects back to `d_model`

The current understanding: **each MLP neuron acts as a detector that fires on specific inputs and writes a specific direction to the residual stream**.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 42, 17, 88, 55])
logits, cache = model.run_with_cache(tokens)
print(f'Model ready. d_mlp = {cfg.d_mlp}')

## Anatomy of a Single MLP Layer

Let's trace exactly what happens inside the MLP in layer 0.

In [ ]:
# The MLP weight matrices
mlp = model.blocks[0].mlp
print(f'W_in shape:  {mlp.W_in.shape}   [d_model, d_mlp]')
print(f'W_out shape: {mlp.W_out.shape}  [d_mlp, d_model]')
print(f'b_in shape:  {mlp.b_in.shape}      [d_mlp]')
print(f'b_out shape: {mlp.b_out.shape}     [d_model]')
print()

# The intermediate activations
pre_act = cache['blocks.0.mlp.hook_pre']   # Before activation fn
post_act = cache['blocks.0.mlp.hook_post']  # After activation fn
print(f'Pre-activation shape:  {pre_act.shape}  [seq, d_mlp]')
print(f'Post-activation shape: {post_act.shape}  [seq, d_mlp]')

In [ ]:
# The activation function creates sparsity
# Many neurons "fire" (have nonzero activation) while others are silent

for pos in range(len(tokens)):
    pre = np.array(pre_act[pos])
    post = np.array(post_act[pos])
    n_active = int(np.sum(np.abs(post) > 0.01))
    total = len(post)
    max_act = float(np.max(np.abs(post)))
    print(f'Position {pos} (token {int(tokens[pos]):2d}): '
          f'{n_active}/{total} neurons active ({n_active/total:.0%}), '
          f'max activation = {max_act:.4f}')

print(f'\nSparsity means each input activates a different subset of neurons.')
print(f'This is how the MLP can store different knowledge in different neurons.')

## Individual Neurons: Detectors and Writers

Each neuron has two aspects:
1. **Input weights** (`W_in[:, neuron]`): What pattern in the residual stream activates this neuron?
2. **Output weights** (`W_out[neuron, :]`): What direction does this neuron write to the residual stream?

We can analyze both sides.

In [ ]:
# Find the most active neuron at position 4 (last token)
activations = np.array(post_act[-1])  # [d_mlp]
top_neurons = np.argsort(np.abs(activations))[::-1][:5]

print(f'Top 5 most active neurons at position 4:')
for rank, neuron in enumerate(top_neurons):
    act = activations[neuron]
    print(f'  #{rank+1}: Neuron {neuron} (activation = {act:.4f})')

# Analyze the top neuron
top_neuron = top_neurons[0]
print(f'\n--- Detailed analysis of neuron {top_neuron} ---')

# Input weights: what activates this neuron?
w_in = np.array(mlp.W_in[:, top_neuron])  # [d_model]

# Which tokens in the vocabulary most strongly activate this neuron?
W_E = np.array(model.embed.W_E)  # [d_vocab, d_model]
token_activations = W_E @ w_in  # [d_vocab]
top_tokens = np.argsort(token_activations)[::-1][:5]
print(f'\nTokens that most activate this neuron (via embedding):')
for tok in top_tokens:
    print(f'  Token {tok}: activation = {token_activations[tok]:.4f}')

In [ ]:
# Output weights: what does this neuron write to the residual stream?
w_out = np.array(mlp.W_out[top_neuron, :])  # [d_model]

# Which tokens does this neuron promote when it fires?
W_U = np.array(model.unembed.W_U)  # [d_model, d_vocab]
logit_effects = w_out @ W_U  # [d_vocab]

# Top promoted tokens
top_promoted = np.argsort(logit_effects)[::-1][:5]
top_suppressed = np.argsort(logit_effects)[:5]

print(f'Neuron {top_neuron} output analysis:')
print(f'\nWhen this neuron fires, it promotes:')
for tok in top_promoted:
    print(f'  Token {tok}: logit effect = {logit_effects[tok]:+.4f}')

print(f'\nAnd suppresses:')
for tok in top_suppressed:
    print(f'  Token {tok}: logit effect = {logit_effects[tok]:+.4f}')

## Using IRTK's Neuron Analysis Tools

IRTK has dedicated tools for neuron analysis.

In [ ]:
from irtk.neurons import (
    neuron_stats, neuron_to_logit, dead_neurons,
)

# Get comprehensive neuron statistics for layer 0
stats = neuron_stats(model, tokens, layer=0)
print(f'Layer 0 neuron statistics:')
print(f'  Active neurons: {stats["n_active"]} / {stats["n_total"]} '
      f'({stats["sparsity"]:.1%} sparse)')
print(f'  Mean activation: {stats["mean_activation"]:.4f}')
print(f'  Max activation:  {stats["max_activation"]:.4f}')

In [ ]:
# Neuron-to-logit mapping: what token does each neuron promote?
n2l = neuron_to_logit(model, layer=0, top_k=3)

print(f'What each neuron promotes (top 3):')
for entry in n2l[:8]:  # Show first 8 neurons
    neuron_id = entry['neuron']
    top_toks = entry['top_positive']
    tok_str = ', '.join([f'tok {t["token"]}({t["logit_effect"]:+.3f})' for t in top_toks])
    print(f'  Neuron {neuron_id:3d}: {tok_str}')

In [ ]:
# Check for dead neurons (never activate)
dead = dead_neurons(model, tokens, layer=0)
print(f'Dead neurons in layer 0: {len(dead["dead_neuron_ids"])} '
      f'out of {cfg.d_mlp}')
if len(dead['dead_neuron_ids']) > 0:
    print(f'  IDs: {dead["dead_neuron_ids"][:10]}...')
print(f'\n(Dead neurons waste model capacity — they respond to nothing.)')

## MLP Ablation: Does This MLP Matter?

Just like with attention heads, we can test an MLP layer's importance by ablating it.

In [ ]:
logits_base = model(tokens)
base_pred = int(jnp.argmax(logits_base[-1]))
base_logit = float(logits_base[-1, base_pred])

print(f'Base prediction: token {base_pred} (logit = {base_logit:.4f})')
print()

for layer in range(cfg.n_layers):
    # Zero out the entire MLP output
    hooks = {f'blocks.{layer}.hook_mlp_out': lambda x, n: jnp.zeros_like(x)}
    logits_abl = model.run_with_hooks(tokens, fwd_hooks=hooks)
    new_logit = float(logits_abl[-1, base_pred])
    change = new_logit - base_logit
    new_pred = int(jnp.argmax(logits_abl[-1]))
    
    print(f'Layer {layer} MLP ablated: logit change = {change:+.4f}, '
          f'new top prediction = token {new_pred}')

    # Also try zeroing just specific neurons
    hooks2 = {
        f'blocks.{layer}.mlp.hook_post': 
        lambda x, n, neuron=top_neurons[0]: x.at[:, neuron].set(0.0)
    }
    logits_abl2 = model.run_with_hooks(tokens, fwd_hooks=hooks2)
    change2 = float(logits_abl2[-1, base_pred]) - base_logit
    print(f'  Single neuron ({top_neurons[0]}) ablated: logit change = {change2:+.4f}')

## The Polysemanticity Problem

A major challenge: in practice, **individual neurons are often polysemantic** — they respond to multiple unrelated concepts. A single neuron might fire for both "the word 'and'" and "mathematical expressions."

This happens because there are far more **features** (concepts the model represents) than there are neurons. The model uses **superposition** — representing more features than it has dimensions by encoding them in overlapping, nearly-orthogonal directions.

**Sparse autoencoders** (Tutorial T10) attempt to disentangle these superposed features into interpretable, monosemantic directions.

In [ ]:
# Illustrate superposition: generate many random inputs and check
# which neurons activate for which inputs
rng = jax.random.PRNGKey(0)

n_examples = 20
activation_counts = np.zeros(cfg.d_mlp)

for i in range(n_examples):
    rng, subkey = jax.random.split(rng)
    random_tokens = jax.random.randint(subkey, (5,), 0, cfg.d_vocab)
    _, random_cache = model.run_with_cache(random_tokens)
    post = np.array(random_cache['blocks.0.mlp.hook_post'][-1])
    activation_counts += (np.abs(post) > 0.01).astype(float)

# How many inputs does each neuron activate for?
print(f'Neuron activation frequency (over {n_examples} random inputs):')
always_active = int(np.sum(activation_counts == n_examples))
never_active = int(np.sum(activation_counts == 0))
sometimes = cfg.d_mlp - always_active - never_active
print(f'  Always active:   {always_active} neurons')
print(f'  Sometimes active: {sometimes} neurons')
print(f'  Never active:     {never_active} neurons')
print(f'\nNeurons that activate for many different inputs may be polysemantic —')
print(f'responding to multiple unrelated features in superposition.')

## Key Takeaways

1. **MLPs process each position independently**, transforming the residual stream
2. **Each neuron is a detector + writer**: input weights determine what it responds to, output weights determine what it writes
3. **Sparsity is key**: only a fraction of neurons fire for any given input
4. **Neurons are often polysemantic**: they respond to multiple concepts due to superposition
5. **Neuron ablation** measures causal importance
6. **Sparse autoencoders** (T10) can disentangle superposed features

**Next: [T06 — Your First Investigation](T06_your_first_investigation.ipynb)** — Putting it all together in a guided end-to-end research project.